# 🎤 Pipecat AI Interview Coach — Real-Time Voice Interaction
### Google Colab Runtime (No Docker Required)

> **What this notebook does**
> 1. Writes every project `.py` / config / client file to disk inside Colab
> 2. Installs all Python & Node dependencies
> 3. Launches the Pipecat back-end server + the config API
> 4. Exposes both servers through public ngrok tunnels
> 5. Builds the Vite front-end and serves it
> 6. Zips the entire project (source + built client) for download

---
### ⚡ Quick-start checklist
| Item | Where to fill it |
|------|-----------------|
| `ELEVENLABS_API_KEY` | **Cell 3 - API Keys** |
| `DEEPGRAM_API_KEY` | **Cell 3 - API Keys** |
| `GROQ_API_KEY` | **Cell 3 - API Keys** |
| `NGROK_AUTH_TOKEN` | **Cell 3 - API Keys** |
| `DAILY_API_KEY` *(optional)* | **Cell 3 - API Keys** |


---
## ✅ Checkpoint 1 — Environment & API Keys


In [ ]:
# @title 🔑 API Keys & Configuration { display-mode: "form" }
import os

# Fill these in before running the notebook
ELEVENLABS_API_KEY  = ""  # @param {type:"string"}
DEEPGRAM_API_KEY    = ""  # @param {type:"string"}
GROQ_API_KEY        = ""  # @param {type:"string"}
NGROK_AUTH_TOKEN    = ""  # @param {type:"string"}
DAILY_API_KEY       = ""  # @param {type:"string"}

BOT_SERVER_PORT    = 7860
CONFIG_SERVER_PORT = 7861
CLIENT_PORT        = 3000

_required = {
    "ELEVENLABS_API_KEY": ELEVENLABS_API_KEY,
    "DEEPGRAM_API_KEY":   DEEPGRAM_API_KEY,
    "GROQ_API_KEY":       GROQ_API_KEY,
    "NGROK_AUTH_TOKEN":   NGROK_AUTH_TOKEN,
}
_missing = [k for k, v in _required.items() if not v.strip()]
if _missing:
    print("WARNING - Empty keys:")
    for k in _missing: print(f"   - {k}")
else:
    print("All required API keys are set.")

os.environ["ELEVENLABS_API_KEY"]  = ELEVENLABS_API_KEY
os.environ["DEEPGRAM_API_KEY"]    = DEEPGRAM_API_KEY
os.environ["GROQ_API_KEY"]        = GROQ_API_KEY
os.environ["DAILY_API_KEY"]       = DAILY_API_KEY
os.environ["CONFIG_SERVER_PORT"]  = str(CONFIG_SERVER_PORT)
os.environ["BOT_SERVER_PORT"]     = str(BOT_SERVER_PORT)


---
## ✅ Checkpoint 2 — Install Dependencies


In [ ]:
# @title 📦 Install system packages
import subprocess, sys

def run(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1000:])
    return r

print("Installing system packages...")
run("apt-get update -qq")
run("apt-get install -y -qq ffmpeg libsndfile1 libportaudio2 nodejs npm curl")

node_v = subprocess.check_output(["node","--version"]).decode().strip()
npm_v  = subprocess.check_output(["npm","--version"]).decode().strip()
print(f"Node.js {node_v}  |  npm {npm_v}")
print("System packages installed.")


In [ ]:
# @title 📦 Install Python packages
import subprocess, sys

pkgs = [
    "aiohttp>=3.13.3",
    "loguru>=0.7.3",
    "python-dotenv",
    "Pillow",
    "pyngrok",
    "pipecat-ai[daily,deepgram,elevenlabs,groq,silero,webrtc,runner]",
]

for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    name = pkg.split("[")[0].split(">=")[0]
    status = "installed" if r.returncode == 0 else "FAILED"
    print(f"  {name}: {status}")

print("\nPython packages done.")


In [ ]:
# @title ✔ Verify imports
import importlib

checks = [
    ("aiohttp", "aiohttp"),
    ("loguru",  "loguru"),
    ("PIL",     "Pillow"),
    ("dotenv",  "python-dotenv"),
    ("pyngrok", "pyngrok"),
]

ok = True
for mod, pkg in checks:
    try:
        importlib.import_module(mod)
        print(f"  OK  {pkg}")
    except ImportError:
        print(f"  FAIL  {pkg}")
        ok = False

try:
    import pipecat.pipeline.pipeline
    print("  OK  pipecat-ai")
except ImportError:
    print("  FAIL  pipecat-ai")
    ok = False

print("\nAll imports verified." if ok else "\nSome imports failed.")


---
## ✅ Checkpoint 3 — Write Project Files to Disk
All server Python files and front-end source files are written here.


In [ ]:
# @title 📁 Create directory structure
import pathlib

PROJECT_ROOT = pathlib.Path("/content/interview_coach")
dirs = [PROJECT_ROOT/"server"/"assets", PROJECT_ROOT/"client"/"src",
        PROJECT_ROOT/"logs"]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  {d}")
print("Directories created.")


In [ ]:
# @title Write server/bot.py
content = "#\n# Copyright (c) 2024-2025, Daily\n# SPDX-License-Identifier: BSD 2-Clause License\n#\n\"\"\"Pipecat AI Interview Coach - bot.py\"\"\"\n\nimport os, json\nfrom dotenv import load_dotenv\nfrom loguru import logger\nfrom PIL import Image\nfrom pipecat.audio.turn.smart_turn.local_smart_turn_v3 import LocalSmartTurnAnalyzerV3\nfrom pipecat.audio.vad.silero import SileroVADAnalyzer\nfrom pipecat.audio.vad.vad_analyzer import VADParams\nfrom pipecat.frames.frames import (\n    BotStartedSpeakingFrame, BotStoppedSpeakingFrame, Frame,\n    LLMRunFrame, OutputImageRawFrame, SpriteFrame,\n)\nfrom pipecat.pipeline.pipeline import Pipeline\nfrom pipecat.pipeline.runner import PipelineRunner\nfrom pipecat.pipeline.task import PipelineParams, PipelineTask\nfrom pipecat.processors.aggregators.llm_context import LLMContext\nfrom pipecat.processors.aggregators.llm_response_universal import LLMContextAggregatorPair\nfrom pipecat.processors.frame_processor import FrameDirection, FrameProcessor\nfrom pipecat.processors.frameworks.rtvi import RTVIObserver, RTVIProcessor\nfrom pipecat.runner.types import DailyRunnerArguments, RunnerArguments, SmallWebRTCRunnerArguments\nfrom pipecat.services.deepgram.stt import DeepgramSTTService\nfrom pipecat.services.elevenlabs.tts import ElevenLabsTTSService\nfrom pipecat.services.groq.llm import GroqLLMService\nfrom pipecat.transports.base_transport import BaseTransport, TransportParams\nfrom pipecat.transports.daily.transport import DailyParams, DailyTransport\nfrom pipecat.transports.smallwebrtc.connection import SmallWebRTCConnection\nfrom pipecat.transports.smallwebrtc.transport import SmallWebRTCTransport\n\nload_dotenv(override=True)\n\nsprites = []\nscript_dir = os.path.dirname(os.path.abspath(__file__))\n\nfor i in range(1, 26):\n    fp = os.path.join(script_dir, f\"assets/robot0{i}.png\")\n    if os.path.exists(fp):\n        with Image.open(fp) as img:\n            sprites.append(OutputImageRawFrame(image=img.tobytes(), size=img.size, format=img.format))\n    else:\n        logger.warning(f\"Asset not found: {fp}\")\n\nquiet_frame = talking_frame = None\nif sprites:\n    sprites.extend(sprites[::-1])\n    quiet_frame   = sprites[0]\n    talking_frame = SpriteFrame(images=sprites)\n\n\nclass TalkingAnimation(FrameProcessor):\n    def __init__(self):\n        super().__init__()\n        self._is_talking = False\n\n    async def process_frame(self, frame: Frame, direction: FrameDirection):\n        await super().process_frame(frame, direction)\n        if talking_frame and isinstance(frame, BotStartedSpeakingFrame):\n            if not self._is_talking:\n                await self.push_frame(talking_frame)\n                self._is_talking = True\n        elif quiet_frame and isinstance(frame, BotStoppedSpeakingFrame):\n            await self.push_frame(quiet_frame)\n            self._is_talking = False\n        await self.push_frame(frame, direction)\n\n\ndef get_config_file_path():\n    return os.path.join(script_dir, \"interview_config.json\")\n\n\ndef load_interview_config():\n    default = {\"botNature\": \"decent\", \"jd\": \"\"}\n    try:\n        fp = get_config_file_path()\n        if os.path.exists(fp):\n            with open(fp, \"r\", encoding=\"utf-8\") as f:\n                cfg = json.load(f)\n            nature = cfg.get(\"botNature\", \"decent\")\n            if nature not in [\"friendly\", \"decent\", \"strict\"]:\n                nature = \"decent\"\n            return {\"botNature\": nature, \"jd\": cfg.get(\"jd\", \"\")}\n        return default\n    except Exception as e:\n        logger.error(f\"Error loading config: {e}\")\n        return default\n\n\ndef save_interview_config(bot_nature, jd):\n    if bot_nature not in [\"friendly\", \"decent\", \"strict\"]:\n        bot_nature = \"decent\"\n    try:\n        with open(get_config_file_path(), \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"botNature\": bot_nature, \"jd\": jd}, f, indent=2, ensure_ascii=False)\n        logger.info(\"Config saved\")\n        return True\n    except Exception as e:\n        logger.error(f\"Error saving config: {e}\")\n        return False\n\n\ndef build_system_prompt(bot_nature=\"decent\", jd=\"\"):\n    MAX_JD = 1500\n    if len(jd) > MAX_JD:\n        jd = jd[:MAX_JD] + \"... [truncated]\"\n\n    traits = {\n        \"friendly\": (\"warm, encouraging, and supportive\",\n                     \"Ask questions in a conversational and friendly manner. Be empathetic.\",\n                     \"Provide positive reinforcement and constructive feedback.\"),\n        \"decent\":   (\"professional, balanced, and respectful\",\n                     \"Ask questions in a professional and fair manner.\",\n                     \"Provide balanced feedback and maintain professional standards.\"),\n        \"strict\":   (\"formal, direct, and challenging\",\n                     \"Ask questions in a rigorous and demanding manner.\",\n                     \"Be direct and hold high standards. Provide critical but fair feedback.\"),\n    }\n    tone, approach, feedback = traits.get(bot_nature.lower(), traits[\"decent\"])\n\n    prompt = (\n        f\"You are an AI interview bot conducting a technical interview. \"\n        f\"Your personality is {tone}.\\n\\n\"\n        f\"Your approach: {approach}\\n\\n\"\n        f\"Feedback style: {feedback}\\n\\n\"\n        \"Important guidelines:\\n\"\n        \"- Your output will be converted to audio so avoid special characters or markdown.\\n\"\n        \"- Keep responses concise and clear.\\n\"\n        \"- Ask one question at a time.\\n\"\n        \"- Listen carefully and follow up when needed.\\n\"\n        \"- Assess technical knowledge, problem-solving, and communication.\\n\"\n        \"- Start by introducing yourself briefly and asking the first question.\"\n    )\n    if jd:\n        prompt += (\n            f\"\\n\\nJob Description:\\n{jd}\\n\\n\"\n            \"Assess the candidate based on the role requirements above.\"\n        )\n    return prompt\n\n\nasync def run_bot(transport: BaseTransport, bot_nature=\"decent\", jd=\"\"):\n    cfg = load_interview_config()\n    bot_nature = cfg.get(\"botNature\", bot_nature)\n    jd = cfg.get(\"jd\", jd)\n    logger.info(f\"Starting bot - nature:{bot_nature} jd_len:{len(jd)}\")\n\n    stt = DeepgramSTTService(api_key=os.getenv(\"DEEPGRAM_API_KEY\"))\n    tts = ElevenLabsTTSService(api_key=os.getenv(\"ELEVENLABS_API_KEY\"), voice_id=\"pNInz6obpgDQGcFmaJgB\")\n    llm = GroqLLMService(api_key=os.getenv(\"GROQ_API_KEY\"))\n\n    context = LLMContext([{\"role\": \"system\", \"content\": build_system_prompt(bot_nature, jd)}])\n    ctx_agg = LLMContextAggregatorPair(context)\n    rtvi = RTVIProcessor()\n    ta   = TalkingAnimation()\n\n    pipeline = Pipeline([\n        transport.input(), rtvi, stt, ctx_agg.user(), llm, tts, ta,\n        transport.output(), ctx_agg.assistant(),\n    ])\n    task = PipelineTask(\n        pipeline,\n        params=PipelineParams(enable_metrics=True, enable_usage_metrics=True),\n        observers=[RTVIObserver(rtvi)],\n    )\n    if quiet_frame:\n        await task.queue_frame(quiet_frame)\n\n    @rtvi.event_handler(\"on_client_ready\")\n    async def on_client_ready(rtvi):\n        await rtvi.set_bot_ready()\n        await task.queue_frames([LLMRunFrame()])\n\n    @transport.event_handler(\"on_client_connected\")\n    async def on_client_connected(transport, client):\n        logger.info(\"Client connected\")\n\n    @transport.event_handler(\"on_client_disconnected\")\n    async def on_client_disconnected(transport, client):\n        logger.info(\"Client disconnected\")\n        await task.cancel()\n\n    await PipelineRunner(handle_sigint=False).run(task)\n\n\nasync def bot(runner_args: RunnerArguments):\n    match runner_args:\n        case DailyRunnerArguments():\n            transport = DailyTransport(\n                runner_args.room_url, runner_args.token, \"Pipecat Bot\",\n                params=DailyParams(\n                    audio_in_enabled=True, audio_out_enabled=True,\n                    video_out_enabled=True, video_out_width=1024, video_out_height=576,\n                    vad_analyzer=SileroVADAnalyzer(params=VADParams(stop_secs=0.2)),\n                    turn_analyzer=LocalSmartTurnAnalyzerV3(),\n                ),\n            )\n        case SmallWebRTCRunnerArguments():\n            transport = SmallWebRTCTransport(\n                webrtc_connection=runner_args.webrtc_connection,\n                params=TransportParams(\n                    audio_in_enabled=True, audio_out_enabled=True,\n                    video_out_enabled=True, video_out_width=1024, video_out_height=576,\n                    vad_analyzer=SileroVADAnalyzer(params=VADParams(stop_secs=0.2)),\n                    turn_analyzer=LocalSmartTurnAnalyzerV3(),\n                ),\n            )\n        case _:\n            logger.error(f\"Unsupported runner type: {type(runner_args)}\")\n            return\n    await run_bot(transport)\n\n\nif __name__ == \"__main__\":\n    import threading\n    from pipecat.runner.run import main\n    from config_server import run_config_server\n\n    def _cs_thread():\n        import asyncio\n        loop = asyncio.new_event_loop()\n        asyncio.set_event_loop(loop)\n        try:\n            loop.run_until_complete(run_config_server())\n        except Exception as e:\n            logger.error(f\"Config server error: {e}\")\n\n    t = threading.Thread(target=_cs_thread, daemon=True, name=\"ConfigServer\")\n    t.start()\n    logger.info(\"Config server thread started\")\n    main()\n"
with open("/content/interview_coach/server/bot.py", "w", encoding="utf-8") as f:
    f.write(content)
print("Written: /content/interview_coach/server/bot.py")


In [ ]:
# @title Write server/config_server.py
content = "import asyncio, json, os\nfrom typing import Optional\nfrom aiohttp import web\nfrom loguru import logger\nfrom bot import save_interview_config\n\nCONFIG_SERVER_PORT = int(os.getenv(\"CONFIG_SERVER_PORT\", \"7861\"))\n\n\nasync def handle_interview_config(request: web.Request) -> web.Response:\n    try:\n        data = await request.json()\n        bot_nature = data.get(\"botNature\", \"decent\")\n        jd = data.get(\"jd\", \"\")\n        valid = [\"friendly\", \"decent\", \"strict\"]\n        if bot_nature not in valid:\n            return web.json_response({\"error\": f\"Invalid botNature. Must be one of: {valid}\"}, status=400)\n        if not jd or not jd.strip():\n            return web.json_response({\"error\": \"jd is required\"}, status=400)\n        success = save_interview_config(bot_nature, jd.strip())\n        if success:\n            return web.json_response({\"success\": True, \"message\": \"Configuration saved successfully\",\n                                      \"botNature\": bot_nature, \"jdLength\": len(jd)})\n        return web.json_response({\"error\": \"Failed to save config\"}, status=500)\n    except json.JSONDecodeError:\n        return web.json_response({\"error\": \"Invalid JSON\"}, status=400)\n    except Exception as e:\n        return web.json_response({\"error\": str(e)}, status=500)\n\n\nasync def handle_health_check(request: web.Request) -> web.Response:\n    return web.json_response({\"status\": \"Ok\", \"service\": \"config-server\"})\n\n\ndef create_app() -> web.Application:\n    app = web.Application()\n\n    async def cors_middleware(app, handler):\n        async def mw(request):\n            resp = web.Response() if request.method == \"OPTIONS\" else await handler(request)\n            allowed = os.getenv(\"ALLOWED_ORIGINS\", \"*\")\n            resp.headers[\"Access-Control-Allow-Origin\"]  = \"*\" if allowed == \"*\" else allowed.split(\",\")[0].strip()\n            resp.headers[\"Access-Control-Allow-Methods\"] = \"GET, POST, OPTIONS\"\n            resp.headers[\"Access-Control-Allow-Headers\"] = \"Content-Type\"\n            return resp\n        return mw\n\n    app.middlewares.append(cors_middleware)\n    app.router.add_post(\"/api/interview-config\", handle_interview_config)\n    app.router.add_get(\"/health\", handle_health_check)\n    return app\n\n\nasync def run_config_server(port: Optional[int] = None):\n    port = port or CONFIG_SERVER_PORT\n    app = create_app()\n    runner = web.AppRunner(app)\n    await runner.setup()\n    host = os.getenv(\"CONFIG_SERVER_HOST\", \"0.0.0.0\")\n    await web.TCPSite(runner, host, port).start()\n    logger.info(f\"Config server on http://{host}:{port}\")\n    try:\n        await asyncio.Event().wait()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        await runner.cleanup()\n\n\nif __name__ == \"__main__\":\n    asyncio.run(run_config_server())\n"
with open("/content/interview_coach/server/config_server.py", "w", encoding="utf-8") as f:
    f.write(content)
print("Written: /content/interview_coach/server/config_server.py")


In [ ]:
# @title Write server/interview_config.json & .env
import json, os

with open("/content/interview_coach/server/interview_config.json", "w") as f:
    json.dump({"botNature": "decent", "jd": ""}, f, indent=2)

env_content = "\n".join([
    f"ELEVENLABS_API_KEY={os.environ.get('ELEVENLABS_API_KEY','')}",
    f"DEEPGRAM_API_KEY={os.environ.get('DEEPGRAM_API_KEY','')}",
    f"GROQ_API_KEY={os.environ.get('GROQ_API_KEY','')}",
    f"DAILY_API_KEY={os.environ.get('DAILY_API_KEY','')}",
    f"CONFIG_SERVER_PORT={os.environ.get('CONFIG_SERVER_PORT','7861')}",
    "CONFIG_SERVER_HOST=0.0.0.0",
    "ALLOWED_ORIGINS=*",
])

with open("/content/interview_coach/server/.env", "w") as f:
    f.write(env_content)

print("Written: server/interview_config.json")
print("Written: server/.env")


In [ ]:
# @title Write client/index.html
content = "<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n  <meta charset=\"UTF-8\" />\n  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />\n  <title>Pipecat AI Interview Coach</title>\n</head>\n<body>\n  <div class=\"container\">\n    <div class=\"header\">\n      <div class=\"transport-selector\">\n        <label for=\"transport-select\">Transport:</label>\n        <select id=\"transport-select\"></select>\n      </div>\n      <div class=\"controls\">\n        <button id=\"mic-btn\" class=\"control-btn\" disabled>\n          \ud83c\udfa4 <span id=\"mic-status\">Mic is Off</span>\n        </button>\n        <button id=\"connect-btn\" class=\"connect-btn\">Connect</button>\n      </div>\n    </div>\n    <div class=\"config-panel\">\n      <div class=\"form-group\">\n        <label for=\"bot-nature-select\">Bot Nature:</label>\n        <select id=\"bot-nature-select\" class=\"form-control\">\n          <option value=\"friendly\">Friendly</option>\n          <option value=\"decent\" selected>Decent</option>\n          <option value=\"strict\">Strict</option>\n        </select>\n      </div>\n      <div class=\"form-group\">\n        <label for=\"jd-textarea\">Job Description (JD):</label>\n        <textarea id=\"jd-textarea\" class=\"form-control\" rows=\"6\"\n          placeholder=\"Enter the job description here (min 50 characters)...\"></textarea>\n        <div class=\"char-count\"><span id=\"char-count\">0</span> characters</div>\n      </div>\n    </div>\n    <div class=\"main-content\">\n      <div class=\"video-panel\">\n        <div id=\"bot-video-container\">\n          <div class=\"video-placeholder\"><span>Video will appear here when connected</span></div>\n        </div>\n      </div>\n      <div class=\"conversation-panel\">\n        <h3>Conversation</h3>\n        <div id=\"conversation-log\"></div>\n      </div>\n    </div>\n    <div class=\"events-panel\">\n      <h3>Events</h3>\n      <div id=\"events-log\"></div>\n    </div>\n  </div>\n  <script type=\"module\" src=\"/src/app.js\"></script>\n  <link rel=\"stylesheet\" href=\"/src/style.css\" />\n</body>\n</html>\n"
with open("/content/interview_coach/client/index.html", "w", encoding="utf-8") as f:
    f.write(content)
print("Written: /content/interview_coach/client/index.html")


In [ ]:
# @title Write client/src/app.js
content = "import { PipecatClient, RTVIEvent } from '@pipecat-ai/client-js';\nimport { AVAILABLE_TRANSPORTS, DEFAULT_TRANSPORT, TRANSPORT_CONFIG, createTransport } from './config';\n\nclass VoiceChatClient {\n  constructor() {\n    this.client = null; this.transportType = DEFAULT_TRANSPORT; this.isConnected = false;\n    this.setupDOM(); this.setupEventListeners(); this.addEvent('initialized', 'Client initialized');\n  }\n\n  setupDOM() {\n    this.transportSelect   = document.getElementById('transport-select');\n    this.connectBtn        = document.getElementById('connect-btn');\n    this.micBtn            = document.getElementById('mic-btn');\n    this.micStatus         = document.getElementById('mic-status');\n    this.conversationLog   = document.getElementById('conversation-log');\n    this.eventsLog         = document.getElementById('events-log');\n    this.botVideoContainer = document.getElementById('bot-video-container');\n    this.jdTextarea        = document.getElementById('jd-textarea');\n    this.botNatureSelect   = document.getElementById('bot-nature-select');\n    this.charCount         = document.getElementById('char-count');\n\n    this.transportSelect.innerHTML = '';\n    AVAILABLE_TRANSPORTS.forEach(t => {\n      const o = document.createElement('option');\n      o.value = t;\n      o.textContent = t === 'smallwebrtc' ? 'SmallWebRTC' : t.charAt(0).toUpperCase() + t.slice(1);\n      this.transportSelect.appendChild(o);\n    });\n    if (AVAILABLE_TRANSPORTS.length === 1) this.transportSelect.parentElement.style.display = 'none';\n    this.addConversationMessage('Connect to start talking with your bot', 'placeholder');\n  }\n\n  setupEventListeners() {\n    this.transportSelect.addEventListener('change', e => {\n      this.transportType = e.target.value;\n      this.addEvent('transport-changed', this.transportType);\n    });\n    this.jdTextarea.addEventListener('input', () => {\n      this.charCount.textContent = this.jdTextarea.value.length;\n    });\n    this.connectBtn.addEventListener('click', () => {\n      if (this.isConnected) { this.disconnect(); }\n      else if (this.validateConfig()) { this.connect(); }\n    });\n    this.micBtn.addEventListener('click', () => {\n      if (this.client) { const s = !this.client.isMicEnabled; this.client.enableMic(s); this.updateMicButton(s); }\n    });\n  }\n\n  validateConfig() {\n    const jd = this.jdTextarea.value.trim();\n    if (!jd) { alert('Please enter a Job Description before starting.'); this.jdTextarea.focus(); return false; }\n    if (jd.length < 50) { alert('Job Description must be at least 50 characters.'); this.jdTextarea.focus(); return false; }\n    return true;\n  }\n\n  getConfig() { return { botNature: this.botNatureSelect.value, jd: this.jdTextarea.value.trim() }; }\n\n  async connect() {\n    try {\n      const cfg = this.getConfig();\n      const configUrl = import.meta.env.VITE_CONFIG_SERVER_URL || 'http://localhost:7861';\n      this.addEvent('connecting', `Using ${this.transportType}`);\n      try {\n        const resp = await fetch(`${configUrl}/api/interview-config`, {\n          method: 'POST', headers: {'Content-Type':'application/json'},\n          body: JSON.stringify({ botNature: cfg.botNature, jd: cfg.jd }),\n        });\n        const result = await resp.json();\n        if (resp.ok) this.addEvent('config-saved', result.message);\n        else throw new Error(result.error || `HTTP ${resp.status}`);\n      } catch (e) { this.addEvent('config-error', e.message); }\n\n      const transport = await createTransport(this.transportType);\n      this.client = new PipecatClient({\n        transport, enableMic: true, enableCam: false,\n        callbacks: {\n          onConnected:             () => this.onConnected(),\n          onDisconnected:          () => this.onDisconnected(),\n          onTransportStateChanged: s  => this.addEvent('transport-state', s),\n          onBotReady:              () => this.addEvent('bot-ready', 'Bot is ready'),\n          onUserTranscript:        d  => { if (d.final) this.addConversationMessage(d.text, 'user'); },\n          onBotTranscript:         d  => this.addConversationMessage(d.text, 'bot'),\n          onError:                 e  => this.addEvent('error', e.message),\n        },\n      });\n      this.setupAudio();\n      await this.client.connect(TRANSPORT_CONFIG[this.transportType]);\n    } catch (e) { this.addEvent('error', e.message); console.error('Connection error:', e); }\n  }\n\n  async disconnect() { if (this.client) await this.client.disconnect(); }\n\n  setupAudio() {\n    this.client.on(RTVIEvent.TrackStarted, (track, participant) => {\n      if (!participant?.local) {\n        if (track.kind === 'audio') {\n          const a = document.createElement('audio'); a.autoplay = true;\n          a.srcObject = new MediaStream([track]); document.body.appendChild(a);\n          this.addEvent('track-started', 'Bot audio');\n        } else if (track.kind === 'video') { this.setupVideoTrack(track); this.addEvent('track-started', 'Bot video'); }\n      }\n    });\n    this.client.on(RTVIEvent.TrackStopped, (track, p) => {\n      if (!p?.local && track.kind === 'video') { this.clearVideoTrack(); }\n    });\n  }\n\n  setupVideoTrack(track) {\n    this.botVideoContainer.innerHTML = '';\n    const v = document.createElement('video');\n    v.autoplay = true; v.playsInline = true; v.muted = true;\n    v.srcObject = new MediaStream([track]);\n    this.botVideoContainer.appendChild(v);\n  }\n\n  clearVideoTrack() {\n    const v = this.botVideoContainer.querySelector('video');\n    if (v?.srcObject) { v.srcObject.getTracks().forEach(t => t.stop()); v.srcObject = null; }\n    this.botVideoContainer.innerHTML = '<div class=\"video-placeholder\"><span>Video will appear here when connected</span></div>';\n  }\n\n  onConnected() {\n    this.isConnected = true; this.connectBtn.textContent = 'Disconnect'; this.connectBtn.classList.add('disconnect');\n    this.micBtn.disabled = false; this.transportSelect.disabled = true;\n    this.updateMicButton(this.client.isMicEnabled); this.addEvent('connected', 'Connected to bot');\n    if (this.conversationLog.querySelector('.placeholder')) this.conversationLog.innerHTML = '';\n  }\n\n  onDisconnected() {\n    this.isConnected = false; this.connectBtn.textContent = 'Connect'; this.connectBtn.classList.remove('disconnect');\n    this.micBtn.disabled = true; this.transportSelect.disabled = false;\n    this.updateMicButton(false); this.clearVideoTrack(); this.addEvent('disconnected', 'Disconnected');\n  }\n\n  updateMicButton(enabled) {\n    this.micStatus.textContent = enabled ? 'Mic is On' : 'Mic is Off';\n    this.micBtn.style.backgroundColor = enabled ? '#10b981' : '#1f2937';\n  }\n\n  addConversationMessage(text, role) {\n    const d = document.createElement('div'); d.className = `conversation-message ${role}`;\n    if (role === 'placeholder') { d.textContent = text; }\n    else {\n      const r = document.createElement('div'); r.className = 'role'; r.textContent = role === 'user' ? 'You' : 'Bot';\n      const t = document.createElement('div'); t.textContent = text;\n      d.appendChild(r); d.appendChild(t);\n    }\n    this.conversationLog.appendChild(d); this.conversationLog.scrollTop = this.conversationLog.scrollHeight;\n  }\n\n  addEvent(name, data) {\n    const d  = document.createElement('div'); d.className = 'event-entry';\n    const ts = document.createElement('span'); ts.className = 'timestamp'; ts.textContent = new Date().toLocaleTimeString();\n    const nm = document.createElement('span'); nm.className = 'event-name'; nm.textContent = name;\n    const dt = document.createElement('span'); dt.className = 'event-data'; dt.textContent = typeof data === 'string' ? data : JSON.stringify(data);\n    d.appendChild(ts); d.appendChild(nm); d.appendChild(dt);\n    this.eventsLog.appendChild(d); this.eventsLog.scrollTop = this.eventsLog.scrollHeight;\n  }\n}\n\nwindow.addEventListener('DOMContentLoaded', () => { new VoiceChatClient(); });\n"
with open("/content/interview_coach/client/src/app.js", "w", encoding="utf-8") as f:
    f.write(content)
print("Written: /content/interview_coach/client/src/app.js")


In [ ]:
# @title Write client/src/config.js
content = "export const AVAILABLE_TRANSPORTS = ['daily', 'smallwebrtc'];\nexport const DEFAULT_TRANSPORT    = 'daily';\n\nconst botStartUrl = import.meta.env.VITE_BOT_START_URL    || 'http://localhost:7860/start';\nconst apiKey      = import.meta.env.VITE_BOT_START_PUBLIC_API_KEY;\nconst _hdr        = () => apiKey ? new Headers({ Authorization: `Bearer ${apiKey}` }) : undefined;\n\nexport const TRANSPORT_CONFIG = {\n  daily: {\n    endpoint: botStartUrl,\n    requestData: { createDailyRoom: true, dailyRoomProperties: { start_video_off: true }, transport: 'daily' },\n    headers: _hdr(),\n  },\n  smallwebrtc: {\n    endpoint: botStartUrl,\n    requestData: { createDailyRoom: false, enableDefaultIceServers: true, transport: 'webrtc' },\n    headers: _hdr(),\n  },\n};\n\nexport async function createTransport(transportType) {\n  switch (transportType) {\n    case 'daily': {\n      const { DailyTransport } = await import('@pipecat-ai/daily-transport');\n      return new DailyTransport();\n    }\n    case 'smallwebrtc': {\n      const { SmallWebRTCTransport } = await import('@pipecat-ai/small-webrtc-transport');\n      return new SmallWebRTCTransport();\n    }\n    default:\n      throw new Error(`Unsupported transport: ${transportType}`);\n  }\n}\n"
with open("/content/interview_coach/client/src/config.js", "w", encoding="utf-8") as f:
    f.write(content)
print("Written: /content/interview_coach/client/src/config.js")


In [ ]:
# @title Write client/src/style.css
content = "* { box-sizing: border-box; }\nbody { margin:0; padding:0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; background:#000; color:#e5e7eb; height:100vh; overflow:hidden; }\n.container { display:flex; flex-direction:column; width:100%; height:100vh; }\n.header { display:flex; justify-content:space-between; align-items:center; padding:1rem; border-bottom:1px solid #333; }\n.transport-selector { display:flex; align-items:center; gap:0.5rem; }\n.transport-selector label { font-weight:500; }\n#transport-select { padding:0.5rem 1rem; background:#1f2937; color:#e5e7eb; border:1px solid #374151; border-radius:0.375rem; cursor:pointer; font-size:0.875rem; }\n#transport-select:focus { outline:none; border-color:#10b981; }\n.controls { display:flex; gap:0.5rem; align-items:center; }\n.control-btn, .connect-btn { padding:0.5rem 1rem; border:none; border-radius:0.375rem; font-size:0.875rem; font-weight:500; cursor:pointer; transition:all 0.2s; }\n.control-btn { background:#1f2937; color:#e5e7eb; border:1px solid #374151; }\n.control-btn:hover:not(:disabled) { background:#374151; }\n.control-btn:disabled { opacity:0.5; cursor:not-allowed; }\n.connect-btn { background:#10b981; color:#000; }\n.connect-btn:hover { background:#059669; }\n.connect-btn.disconnect { background:#ef4444; color:#fff; }\n.connect-btn.disconnect:hover { background:#dc2626; }\n.config-panel { padding:1rem; border-bottom:1px solid #333; display:flex; gap:1rem; flex-wrap:wrap; }\n.form-group { display:flex; flex-direction:column; gap:0.25rem; flex:1; min-width:200px; }\n.form-group label { font-size:0.875rem; font-weight:500; color:#9ca3af; }\n.form-control { padding:0.5rem; background:#1f2937; color:#e5e7eb; border:1px solid #374151; border-radius:0.375rem; font-size:0.875rem; }\n.form-control:focus { outline:none; border-color:#10b981; }\ntextarea.form-control { resize:vertical; }\n.char-count { font-size:0.75rem; color:#6b7280; text-align:right; margin-top:0.25rem; }\n.main-content { flex:1; display:flex; overflow:hidden; }\n.video-panel { flex:1; padding:1rem; display:flex; flex-direction:column; border-right:1px solid #333; }\n#bot-video-container { flex:1; background:#111; border:1px solid #333; border-radius:0.5rem; overflow:hidden; display:flex; align-items:center; justify-content:center; }\n#bot-video-container video { width:100%; height:100%; object-fit:cover; }\n.video-placeholder { color:#6b7280; font-style:italic; text-align:center; padding:2rem; }\n.conversation-panel { flex:1; padding:1rem; overflow:hidden; display:flex; flex-direction:column; }\n.conversation-panel h3 { margin:0 0 1rem; font-size:1rem; font-weight:600; }\n#conversation-log { flex:1; overflow-y:auto; background:#111; border:1px solid #333; border-radius:0.5rem; padding:1rem; font-size:0.875rem; line-height:1.5; }\n.conversation-message { margin-bottom:0.75rem; padding:0.5rem; border-radius:0.25rem; }\n.conversation-message.user { background:#1e40af; color:#e0e7ff; }\n.conversation-message.bot  { background:#065f46; color:#d1fae5; }\n.conversation-message .role { font-weight:600; margin-bottom:0.25rem; }\n.conversation-message.placeholder { color:#6b7280; font-style:italic; background:transparent; }\n.events-panel { height:256px; padding:0 1rem 1rem; border-top:1px solid #333; }\n.events-panel h3 { margin:1rem 0; font-size:1rem; font-weight:600; }\n#events-log { height:calc(100% - 3rem); overflow-y:auto; background:#111; border:1px solid #333; border-radius:0.5rem; padding:0.75rem; font-family:'Courier New',monospace; font-size:0.75rem; line-height:1.4; }\n.event-entry { margin-bottom:0.5rem; display:flex; gap:0.5rem; }\n.event-entry .timestamp  { color:#6b7280; flex-shrink:0; }\n.event-entry .event-name { color:#10b981; font-weight:600; flex-shrink:0; }\n.event-entry .event-data { color:#9ca3af; }\n#conversation-log::-webkit-scrollbar, #events-log::-webkit-scrollbar { width:8px; }\n#conversation-log::-webkit-scrollbar-track, #events-log::-webkit-scrollbar-track { background:#1f2937; border-radius:4px; }\n#conversation-log::-webkit-scrollbar-thumb, #events-log::-webkit-scrollbar-thumb { background:#374151; border-radius:4px; }\n"
with open("/content/interview_coach/client/src/style.css", "w", encoding="utf-8") as f:
    f.write(content)
print("Written: /content/interview_coach/client/src/style.css")


In [ ]:
# @title Write client/package.json & vite.config.js
import json

pkg = {
  "name": "interview-coach-client", "private": True, "version": "0.0.1", "type": "module",
  "scripts": {"dev": "vite", "build": "vite build", "preview": "vite preview"},
  "dependencies": {
    "@pipecat-ai/client-js": "^1.5.0",
    "@pipecat-ai/daily-transport": "^1.5.0",
    "@pipecat-ai/small-webrtc-transport": "^1.8.0"
  },
  "devDependencies": {"vite": "^7.1.7"}
}
with open("/content/interview_coach/client/package.json", "w") as f:
    json.dump(pkg, f, indent=2)

vite_cfg = "import { defineConfig } from 'vite'\nexport default defineConfig({\n  server: { port: 3000, host: '0.0.0.0', cors: true },\n  build:  { outDir: 'dist', sourcemap: false },\n  define: { 'process.env': {} }\n})\n"
with open("/content/interview_coach/client/vite.config.js", "w") as f:
    f.write(vite_cfg)

print("Written: client/package.json")
print("Written: client/vite.config.js")


In [ ]:
# @title Verify all project files
import os

files = [
    "/content/interview_coach/server/bot.py",
    "/content/interview_coach/server/config_server.py",
    "/content/interview_coach/server/interview_config.json",
    "/content/interview_coach/server/.env",
    "/content/interview_coach/client/package.json",
    "/content/interview_coach/client/vite.config.js",
    "/content/interview_coach/client/index.html",
    "/content/interview_coach/client/src/app.js",
    "/content/interview_coach/client/src/config.js",
    "/content/interview_coach/client/src/style.css",
]
all_ok = True
for fp in files:
    ok = os.path.exists(fp)
    sz = os.path.getsize(fp) if ok else 0
    print(f"  {'OK' if ok else 'MISSING':7s} {os.path.basename(fp):35s} {sz:,} bytes")
    all_ok = all_ok and ok
print("\nAll files verified." if all_ok else "\nSome files missing - re-run cells above.")


---
## ✅ Checkpoint 4 — Start ngrok Tunnels


In [ ]:
# @title Start ngrok tunnels
from pyngrok import ngrok, conf
import time

conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
time.sleep(1)

bot_tunnel    = ngrok.connect(BOT_SERVER_PORT,    "http")
config_tunnel = ngrok.connect(CONFIG_SERVER_PORT, "http")

BOT_PUBLIC_URL    = bot_tunnel.public_url.replace("http://","https://")
CONFIG_PUBLIC_URL = config_tunnel.public_url.replace("http://","https://")

print(f"Bot server  -> {BOT_PUBLIC_URL}")
print(f"Config API  -> {CONFIG_PUBLIC_URL}")


In [ ]:
# @title Write client/.env.local with live ngrok URLs
env_local = f"""VITE_BOT_START_URL={BOT_PUBLIC_URL}/start
VITE_CONFIG_SERVER_URL={CONFIG_PUBLIC_URL}
"""
with open("/content/interview_coach/client/.env.local", "w") as f:
    f.write(env_local)
print("client/.env.local written")
print(env_local)


---
## ✅ Checkpoint 5 — Generate Robot Animation Assets


In [ ]:
# @title Generate 25 placeholder robot frames
from PIL import Image, ImageDraw
import os, math

ASSET_DIR = "/content/interview_coach/server/assets"
os.makedirs(ASSET_DIR, exist_ok=True)
W, H = 1024, 576

def make_frame(i, total=25):
    p = i / total
    img = Image.new("RGB", (W, H), (10, 10, 20))
    draw = ImageDraw.Draw(img)
    for x in range(W):
        r = int(20 + 30 * math.sin((x/W + p) * math.pi))
        g = int(40 + 20 * math.cos((x/W + p) * 2 * math.pi))
        draw.line([(x,0),(x,H)], fill=(r,g,60))
    draw.rectangle([362,180,662,460], fill=(30,30,50), outline=(16,185,129), width=3)
    ey = 30 if i < 20 else 5
    draw.rectangle([410,240,460,240+ey], fill=(16,185,129))
    draw.rectangle([562,240,612,240+ey], fill=(16,185,129))
    mw = int(40 + 40*math.sin(p*math.pi))
    draw.rectangle([512-mw,350,512+mw,375], fill=(16,185,129))
    draw.text((20,20), f"Robot {i+1:02d}", fill=(100,200,150))
    return img

for i in range(25):
    make_frame(i).save(os.path.join(ASSET_DIR, f"robot0{i+1}.png"), "PNG")
print(f"Generated 25 robot frames in {ASSET_DIR}")
print("Replace with original assets for real robot avatar.")


---
## ✅ Checkpoint 6 — Start Back-End Servers


In [ ]:
# @title Start config server and bot server
import subprocess, sys, time, os, pathlib

SERVER_DIR = "/content/interview_coach/server"
LOG_DIR    = "/content/interview_coach/logs"
pathlib.Path(LOG_DIR).mkdir(exist_ok=True)

env = os.environ.copy()
env.update({
    "PYTHONPATH":         SERVER_DIR,
    "PYTHONUNBUFFERED":   "1",
    "CONFIG_SERVER_PORT": str(CONFIG_SERVER_PORT),
    "ELEVENLABS_API_KEY": ELEVENLABS_API_KEY,
    "DEEPGRAM_API_KEY":   DEEPGRAM_API_KEY,
    "GROQ_API_KEY":       GROQ_API_KEY,
    "DAILY_API_KEY":      DAILY_API_KEY,
    "ALLOWED_ORIGINS":    "*",
})

config_log  = open(f"{LOG_DIR}/config_server.log", "w")
config_proc = subprocess.Popen(
    [sys.executable, f"{SERVER_DIR}/config_server.py"],
    stdout=config_log, stderr=config_log, env=env, cwd=SERVER_DIR)
print(f"Config server PID: {config_proc.pid}")

time.sleep(3)

bot_log  = open(f"{LOG_DIR}/bot_server.log", "w")
bot_proc = subprocess.Popen(
    [sys.executable, "-m", "pipecat.runner.run",
     "--host", "0.0.0.0", "--port", str(BOT_SERVER_PORT),
     "--bot-file", f"{SERVER_DIR}/bot.py"],
    stdout=bot_log, stderr=bot_log, env=env, cwd=SERVER_DIR)
print(f"Bot server     PID: {bot_proc.pid}")

time.sleep(5)

import urllib.request, json as _j
try:
    r = urllib.request.urlopen(f"http://localhost:{CONFIG_SERVER_PORT}/health", timeout=4)
    print(f"Config server health: {_j.loads(r.read())}")
except Exception as e:
    print(f"Config server not yet ready: {e}")


In [ ]:
# @title Tail server logs (run anytime to debug)
print("=== Config Server (last 20 lines) ===")
!tail -20 /content/interview_coach/logs/config_server.log
print("\n=== Bot Server (last 20 lines) ===")
!tail -20 /content/interview_coach/logs/bot_server.log


---
## ✅ Checkpoint 7 — Build the Front-End Client


In [ ]:
# @title npm install
import subprocess

r = subprocess.run(["npm","install"], cwd="/content/interview_coach/client", capture_output=True, text=True)
print(r.stdout[-2000:] if r.stdout else "")
if r.returncode != 0:
    print("STDERR:", r.stderr[-1000:])
    raise RuntimeError("npm install failed")
print("npm packages installed.")


In [ ]:
# @title vite build
import subprocess, os

r = subprocess.run(["npm","run","build"], cwd="/content/interview_coach/client", capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise RuntimeError("vite build failed")

dist_files = []
for root, _, files in os.walk("/content/interview_coach/client/dist"):
    for f in files: dist_files.append(os.path.join(root,f))
print(f"Build succeeded - {len(dist_files)} output files in client/dist/")


In [ ]:
# @title Serve built client via ngrok
from pyngrok import ngrok as _ng
import subprocess, sys, time, pathlib

pathlib.Path("/content/interview_coach/logs").mkdir(exist_ok=True)
client_log  = open("/content/interview_coach/logs/client.log", "w")
client_proc = subprocess.Popen(
    [sys.executable, "-m", "http.server", str(CLIENT_PORT),
     "--directory", "/content/interview_coach/client/dist"],
    stdout=client_log, stderr=client_log)
print(f"Static server PID: {client_proc.pid}")
time.sleep(2)

client_tunnel = _ng.connect(CLIENT_PORT, "http")
CLIENT_PUBLIC_URL = client_tunnel.public_url.replace("http://","https://")
print(f"Client URL -> {CLIENT_PUBLIC_URL}")
print("\nOpen the URL above in your browser to use the Interview Coach!")


---
## ✅ Checkpoint 8 — Live Summary


In [ ]:
# @title Print live URLs and process status
procs = {
    "Config Server (7861)": config_proc,
    "Bot Server    (7860)": bot_proc,
    "Client Server (3000)": client_proc,
}
print("="*60)
print("  PIPECAT AI INTERVIEW COACH -- LIVE")
print("="*60)
print(f"\n  Client UI  -> {CLIENT_PUBLIC_URL}")
print(f"  Bot API    -> {BOT_PUBLIC_URL}")
print(f"  Config API -> {CONFIG_PUBLIC_URL}")
print("\nProcesses:")
for name, proc in procs.items():
    poll = proc.poll()
    st = "Running" if poll is None else f"Exited ({poll})"
    print(f"  {name}: {st}  (PID {proc.pid})")


---
## ✅ Checkpoint 9 — Zip & Download All Output


In [ ]:
# @title Create output zip
import zipfile, os, pathlib, datetime

TIMESTAMP  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ZIP = f"/content/pipecat_interview_coach_{TIMESTAMP}.zip"

INCLUDE = [
    "/content/interview_coach/server",
    "/content/interview_coach/client/dist",
    "/content/interview_coach/client/src",
    "/content/interview_coach/client/index.html",
    "/content/interview_coach/client/package.json",
    "/content/interview_coach/client/vite.config.js",
    "/content/interview_coach/logs",
]

SKIP = {"node_modules", ".git", "__pycache__", ".venv"}

def should_skip(p):
    return any(s in pathlib.Path(p).parts for s in SKIP) or str(p).endswith(".pyc")

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in INCLUDE:
        p = pathlib.Path(item)
        if p.is_file():
            zf.write(p, p.relative_to("/content"))
        elif p.is_dir():
            for root, dirs, files in os.walk(p):
                dirs[:] = [d for d in dirs if d not in SKIP]
                for f in files:
                    fp = pathlib.Path(root)/f
                    if not should_skip(fp):
                        zf.write(fp, fp.relative_to("/content"))

sz = os.path.getsize(OUTPUT_ZIP)/1_048_576
with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    n = len(zf.namelist())
print(f"Zip created: {OUTPUT_ZIP}")
print(f"Size: {sz:.2f} MB  |  Files: {n}")


In [ ]:
# @title Download zip to your computer
from google.colab import files
print("Initiating download...")
files.download(OUTPUT_ZIP)
print("Download triggered.")


---
## 🛑 Optional — Stop All Servers


In [ ]:
# @title Stop all servers and tunnels
from pyngrok import ngrok as _ng

for name, proc in [("Config server", config_proc), ("Bot server", bot_proc), ("Client server", client_proc)]:
    if proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"Stopped: {name}")
    else:
        print(f"Already stopped: {name}")

_ng.kill()
print("ngrok tunnels closed")
